In [13]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from the same folder as your notebook

api_key = os.getenv("ANTHROPIC_API_KEY")

In [14]:
#create an API client
from anthropic import Anthropic
model = "claude-sonnet-4-5"

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [20]:
def add_user_message(messages, text):
    user_message = {"role": "user" , "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant" , "content": text}
    messages.append(assistant_message)
    
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences or []
    }
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text

In [9]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

# stream = client.messages.create(
#     model=model,
#     max_tokens=1000,
#     messages=messages,
#     stream=True
# )

# for event in stream:
#     print(event)
    
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()
    
final_message.content[0].text

'A fake database is a simulated or mock data storage system that mimics the structure and behavior of a real database but contains fabricated, synthetic, or placeholder data used for testing, development, or demonstration purposes.'

In [21]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

text

'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [22]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())